# Trabalho 1 — TDD e propriedades em pipelines de ML
## Testes Automatizados — IEC PUC Minas

**Aluno:** Andre Cardoso de Oliveira

Notebook no mesmo formato das aulas (Colab + `ipytest` + `hypothesis`).

O enunciado pede 4 elementos. Este notebook cobre os quatro:

1. **TDD** (Red → Green → Refactor) — relato visível nas células, como na Aula 2.
2. **Teste de propriedade** — invariante com Hypothesis, para qualquer entrada válida.
3. **Versão bugada** — a mesma propriedade falha contra uma função propositalmente errada.
4. **Decisões de design** — seção 4, no fim.


## Função escolhida

`precision(y_true, y_pred)` — **precisão binária** na avaliação de um classificador.

No pipeline de ML isso entra depois da predição: das vezes em que o modelo disse “positivo” (`y_pred == 1`), quantas estavam certas (`y_true == 1`)?

```
precision = TP / (TP + FP)
```

Classe positiva = `1`. É uma métrica de avaliação, não de pré-processamento 

**Decisão de design (divisão por zero):** se o modelo não prevê nenhum positivo (`TP + FP == 0`), devolvemos `0.0`. Não crasha e não devolve `NaN`. Motivo: “não acertou nenhum positivo previsto” é precisão zero, não um erro de runtime. Fica documentada e coberta por teste no Green.


In [ ]:
!pip install -q "ipytest==0.14.*" "hypothesis==6.164.*"

import ipytest
import pytest
ipytest.autoconfig()


---
## 1. TDD — `precision`

Ciclo da Aula 2: **vermelho → verde → refatoração**. Um teste por vez.


### 🔴 Passo 1 — Red

O teste é escrito **antes** da função existir. Deve falhar (`precision` ainda não existe). Isso não é erro: é o ponto de partida do TDD.

Caso: verdade `[1, 0, 1, 0, 1]`, predito `[1, 1, 1, 0, 0]`.

- TP = 2 (índices 0 e 2)
- FP = 1 (índice 1)
- precisão = `2 / 3`


In [ ]:
%%ipytest

# garante o estado inicial do TDD mesmo se você já rodou células mais à frente
try:
    del precision
except NameError:
    pass


def test_precision_on_mixed_predictions():
    y_true = [1, 0, 1, 0, 1]
    y_pred = [1, 1, 1, 0, 0]
    assert precision(y_true, y_pred) == 2 / 3


### Relato Red

Rode a célula acima. O relatório do `pytest` deve mostrar **FAILED** / `NameError: name 'precision' is not defined`.

**Não implemente a função ainda.** Este estado (teste existe, código ainda não) é o Red.

A seção 2 (propriedade) está abaixo. Rode essa célula só depois do Green.


### 🟢 Passo 2 — Green (o mínimo possível)

Implementação propositalmente incompleta (*fake it*): só o suficiente para passar **este** teste. `return 2 / 3` funciona para o caso acima, mas ainda não é a função de verdade.


In [ ]:
%%ipytest

def precision(y_true, y_pred):
    return 2 / 3


def test_precision_on_mixed_predictions():
    y_true = [1, 0, 1, 0, 1]
    y_pred = [1, 1, 1, 0, 0]
    assert precision(y_true, y_pred) == 2 / 3


### 🔴🟢 Um novo teste força a implementação real

Com o `return 2 / 3`, o primeiro teste continua passando. O novo caso é um classificador que só acerta os positivos que prevê — a implementação falsa não sobrevive.


In [ ]:
%%ipytest

def precision(y_true, y_pred):
    return 2 / 3


def test_precision_on_mixed_predictions():
    y_true = [1, 0, 1, 0, 1]
    y_pred = [1, 1, 1, 0, 0]
    assert precision(y_true, y_pred) == 2 / 3


def test_precision_when_every_positive_prediction_is_correct():
    y_true = [1, 1, 0]
    y_pred = [1, 1, 0]
    assert precision(y_true, y_pred) == 1.0


### 🟢 Implementação real

Agora a lógica de verdade: contar TP e FP e dividir. Sem positivo previsto, devolve `0.0` — a decisão de divisão por zero, coberta pelo terceiro teste.


In [ ]:
%%ipytest

def precision(y_true, y_pred):
    true_positives = 0
    false_positives = 0
    for expected, predicted in zip(y_true, y_pred):
        if predicted == 1:
            if expected == 1:
                true_positives += 1
            else:
                false_positives += 1
    predicted_positive_count = true_positives + false_positives
    if predicted_positive_count == 0:
        return 0.0
    return true_positives / predicted_positive_count


def test_precision_on_mixed_predictions():
    y_true = [1, 0, 1, 0, 1]
    y_pred = [1, 1, 1, 0, 0]
    assert precision(y_true, y_pred) == 2 / 3


def test_precision_when_every_positive_prediction_is_correct():
    y_true = [1, 1, 0]
    y_pred = [1, 1, 0]
    assert precision(y_true, y_pred) == 1.0


def test_precision_when_there_are_no_predicted_positives():
    y_true = [1, 0, 0]
    y_pred = [0, 0, 0]
    assert precision(y_true, y_pred) == 0.0


### Relato Green

1. Célula *fake it* (`return 2 / 3`): o teste Red passa — **1 passed**. Ainda não é a função real.
2. Célula com o segundo caso (`[1, 1, 0]` / `[1, 1, 0]` → `1.0`): o fake quebra — **FAILED**.
3. Célula da implementação real (conta TP/FP): os três testes passam, inclusive o de nenhum positivo previsto.

Depois disso, a célula de **propriedade** (abaixo) já pode rodar: a função existe, e o Hypothesis vai gerar muitos pares de listas.


### 🔵 Passo 3 — Refactor

Só refatoramos com os testes verdes. A lógica não muda: checamos o comprimento, nomeamos o denominador e documentamos o contrato (incluindo a divisão por zero). Os três testes do Green têm que continuar passando.


In [ ]:
%%ipytest

def precision(y_true: list[int], y_pred: list[int]) -> float:
    """Precisão binária (classe positiva = 1): TP / (TP + FP).

    Se ninguém foi previsto como positivo, devolve 0.0 (não divide por zero).
    """
    if len(y_true) != len(y_pred):
        raise ValueError("y_true e y_pred precisam ter o mesmo comprimento")

    true_positives = 0
    false_positives = 0
    for expected, predicted in zip(y_true, y_pred):
        if predicted == 1:
            if expected == 1:
                true_positives += 1
            else:
                false_positives += 1

    predicted_positive_count = true_positives + false_positives
    if predicted_positive_count == 0:
        return 0.0
    return true_positives / predicted_positive_count


def test_precision_on_mixed_predictions():
    y_true = [1, 0, 1, 0, 1]
    y_pred = [1, 1, 1, 0, 0]
    assert precision(y_true, y_pred) == 2 / 3


def test_precision_when_every_positive_prediction_is_correct():
    y_true = [1, 1, 0]
    y_pred = [1, 1, 0]
    assert precision(y_true, y_pred) == 1.0


def test_precision_when_there_are_no_predicted_positives():
    y_true = [1, 0, 0]
    y_pred = [0, 0, 0]
    assert precision(y_true, y_pred) == 0.0


def test_precision_rejects_mismatched_lengths():
    with pytest.raises(ValueError):
        precision([1, 0], [1])


### Relato Refactor

A célula acima deve mostrar **4 passed**. Se algum teste falhasse, o refactor teria mudado o comportamento — e aí voltaríamos atrás.

O que mudou (clareza e contrato):

- docstring com a fórmula e a decisão de divisão por zero
- `ValueError` quando os comprimentos diferem
- tipos `list[int] -> float`
- teste extra cobrindo o comprimento inválido

O que **não** mudou: para as mesmas entradas válidas, as mesmas saídas.


---
## 2. Teste de propriedade (invariante)

O teste Red acima é **um exemplo pontual**: para aquele par de listas, esperamos `2 / 3`.

Um teste de propriedade troca essa pergunta por outra: **para qualquer entrada válida, esta afirmação é sempre verdadeira?** O `hypothesis` gera dezenas de pares sozinho e procura um contraexemplo (e reduz ao caso mais simples — *shrinking*).

### Invariante escolhida

> A precisão **sempre cai no intervalo fechado [0, 1]**.

Em código: `0.0 <= precision(y_true, y_pred) <= 1.0`.

Isso vale no caso pontual (`2 / 3`), no acerto total (`1.0`), no “ninguém previsto positivo” (`0.0`) e em qualquer par de listas binárias do mesmo tamanho.

Não é um exemplo disfarçado: não fixamos TP nem FP. O gerador monta os rótulos.

**Domínio válido:** listas de `0` e `1`, mesmo comprimento, pelo menos 1 elemento (`min_size=1`). Fora disso a função nem promete número — comprimentos diferentes levantam `ValueError`.

**Quando rodar:** só depois do Green. Sem a função, este teste também falha — e aí não estamos testando a propriedade, só a ausência do código.


In [ ]:
%%ipytest

from hypothesis import given, strategies as st

# Rode depois do Green. A função precisa existir.


@given(
    st.lists(
        st.tuples(st.sampled_from([0, 1]), st.sampled_from([0, 1])),
        min_size=1,
        max_size=40,
    )
)
def test_precision_always_in_unit_interval(pairs):
    y_true = [expected for expected, _ in pairs]
    y_pred = [predicted for _, predicted in pairs]
    result = precision(y_true, y_pred)
    assert 0.0 <= result <= 1.0


---
## 3. A propriedade pega um bug real

A mesma invariante, agora contra uma versão **errada**.

O bug é plausível em produção: alguém escreve a fórmula `TP / (TP + FP)` e **esquece o caso TP + FP == 0**. Em Python isso vira `ZeroDivisionError` no meio de um job de avaliação — ou, em outras linguagens, `Inf`/`NaN` que contaminam o dashboard.

O teste de exemplo `[1, 0, 0]` / `[0, 0, 0]` também falharia, mas o ponto aqui é outro: a propriedade não precisa desse exemplo. Ela gera pares até achar um em que ninguém foi previsto positivo e quebra.

Rode a célula. Esperado: **FAILED**, com um contraexemplo do Hypothesis (ou `ZeroDivisionError`) — não `NameError`.


In [ ]:
%%ipytest

from hypothesis import given, strategies as st


def precision_buggy(y_true, y_pred):
    true_positives = 0
    false_positives = 0
    for expected, predicted in zip(y_true, y_pred):
        if predicted == 1:
            if expected == 1:
                true_positives += 1
            else:
                false_positives += 1
    return true_positives / (true_positives + false_positives)


@given(
    st.lists(
        st.tuples(st.sampled_from([0, 1]), st.sampled_from([0, 1])),
        min_size=1,
        max_size=40,
    )
)
def test_buggy_version_always_in_unit_interval(pairs):
    y_true = [expected for expected, _ in pairs]
    y_pred = [predicted for _, predicted in pairs]
    result = precision_buggy(y_true, y_pred)
    assert 0.0 <= result <= 1.0


### Relato — o teste teria pego o bug

A célula acima deve falhar. O Hypothesis encontra um par em que `y_pred` não contém `1` (por exemplo `[0]` / `[0]`, ou verdade `[1]` e predito `[0]`) e o relatório mostra `ZeroDivisionError`.

Isso é a prova pedida: a propriedade, sozinha, detecta a função bugada. Em produção, um `return tp / (tp + fp)` sem guarda não passaria nessa suíte.


---
## 4. Decisões de design

Aplicáveis a `precision`. O que o enunciado cita e **não** cabe aqui fica explícito.

| Tópico | Decisão | Por quê |
|---|---|---|
| **Divisão por zero** | `TP + FP == 0` → `0.0`. | Sem positivo previsto não há “taxa de acerto entre os previstos”. Crash ou `NaN` quebraria o pipeline de métricas. Coberto pelo teste Green `[1, 0, 0]` / `[0, 0, 0]`. |
| **Comprimentos diferentes** | `ValueError`. | Não dá para alinhar verdade e predito. Não preenche com sentinela. |
| **Lista vazia** | Entrada inválida (`min_size=1` no Hypothesis). | Não existe precisão de um lote sem exemplos. |
| **Empate** | Não se aplica. Não há ranking nem `argmax`. | — |
| **NaN / inf** | Fora do domínio. Rótulos são `0` ou `1`, não scores. | Quem avalia compara classes, não probabilidades. Validar score é etapa **antes** (treino/inferência), não nesta métrica. |
| **Outros inteiros (`2`, `-1`)** | Tratados como “não positivo”. Só `1` conta como classe positiva. | Contrato binário simples; um rótulo multi-classe mal passado não vira positivo por acidente. |
| **`bool`** | `True` conta como `1`, `False` como `0` (é o comportamento de `== 1` em Python). | Aceitável aqui porque o contrato já é binário. Não convertemos string `"1"`. |
| **String `"1"`** | Não é positivo (`"1" == 1` é `False`). | Validar ≠ converter; se o CSV chegou como texto, o erro é de quem carregou o lote. |

A propriedade (`0 <= precisão <= 1`) **não** distingue `2/3` de `1/2` — os dois estão no intervalo. Os casos pontuais do Green é que fixam a fórmula. A divisão por zero é decisão extra, documentada, coberta por teste de exemplo **e** pela propriedade (a versão bugada explode exatamente aí).
